# Population Exposure From Event BBoxes

This notebook:
- reads event metadata from `beliefs/events.asl`
- matches each event to the appropriate WorldPop raster in `Population_data/`
- crops the raster to each event bbox
- computes population exposure metrics
- exports CSV and ASL facts for the reasoning layer

In [13]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from shapely.geometry import box, mapping
from pyproj import Geod

In [14]:
# Project paths
ROOT = Path.cwd()
EVENTS_PATH = ROOT / 'beliefs' / 'events.asl'
POP_DIR = ROOT / 'Population_data'
CROPS_DIR = ROOT / 'Population_data' / 'cropped_to_events'
OUT_CSV = ROOT /'population_exposure_summary.csv'
OUT_ASL = ROOT / 'beliefs' / 'population_exposure.asl'

CROPS_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_ASL.parent.mkdir(parents=True, exist_ok=True)

ROOT, EVENTS_PATH, POP_DIR

(WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain'),
 WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/beliefs/events.asl'),
 WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/Population_data'))

In [15]:
# Parse events, countries, and bboxes from events.asl
text = EVENTS_PATH.read_text(encoding='utf-8')

events = sorted(set(re.findall(r'^event\(([^)]+)\)\.$', text, flags=re.MULTILINE)))
country_map = {e: c for e, c in re.findall(r'^country\(([^,]+),\s*([^)]+)\)\.$', text, flags=re.MULTILINE)}
bbox_map = {
    e: tuple(map(float, (minx, miny, maxx, maxy)))
    for e, minx, miny, maxx, maxy in re.findall(
        r'^bbox\(([^,]+),\s*([\-0-9.]+),\s*([\-0-9.]+),\s*([\-0-9.]+),\s*([\-0-9.]+)\)\.$',
        text,
        flags=re.MULTILINE,
    )
}

parsed = []
for e in events:
    if e in country_map and e in bbox_map:
        parsed.append({
            'event': e,
            'country': country_map[e],
            'bbox': bbox_map[e],
        })

pd.DataFrame(parsed)

,event,country,bbox
0,ahr_valley,germany,"(6.950227, 50.471351, 7.18725, 50.566012)"
1,antalya,turkey,"(30.938848, 36.610847, 32.116923, 37.114228)"
2,emilia,italy,"(11.820598, 44.502971, 12.032391, 44.573716)"
3,evros,greece,"(25.144957, 40.781609, 26.258694, 41.221014)"
4,serra_de_estrela,portugal,"(-7.764595, 40.300913, -7.196839, 40.547715)"
5,sindh,pakistan,"(67.67564, 27.258053, 68.528303, 27.914313)"


In [16]:
# Build country -> raster mapping from file names
# Expected naming like: deu_pop_2021_CN_100m_R2025A_v1.tif
raster_by_iso3 = {}
for tif in sorted(POP_DIR.glob('*.tif')):
    iso3 = tif.name[:3].lower()
    raster_by_iso3[iso3] = tif

country_to_iso3 = {
    'germany': 'deu',
    'italy': 'ita',
    'pakistan': 'pak',
    'greece': 'grc',
    'turkey': 'tur',
    'portugal': 'prt',
}

raster_by_iso3

{'deu': WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/Population_data/deu_pop_2021_CN_100m_R2025A_v1.tif'),
 'grc': WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/Population_data/grc_pop_2023_CN_100m_R2025A_v1.tif'),
 'ita': WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/Population_data/ita_pop_2023_CN_100m_R2025A_v1.tif'),
 'pak': WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/Population_data/pak_pop_2022_CN_100m_R2025A_v1.tif'),
 'prt': WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/Population_data/prt_pop_2022_CN_100m_R2025A_v1.tif'),
 'tur': WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/Population_data/tur_pop_2021_CN_100m_R2025A_v1.tif')}

In [17]:
# Compute population metrics and save per-event raster crop
geod = Geod(ellps='WGS84')
rows = []

for rec in parsed:
    event_id = rec['event']
    country = rec['country']
    minx, miny, maxx, maxy = rec['bbox']

    iso3 = country_to_iso3.get(country)
    if iso3 is None or iso3 not in raster_by_iso3:
        print(f'Skipping {event_id}: no raster found for country={country}')
        continue

    pop_tif = raster_by_iso3[iso3]
    geom_wgs84 = box(minx, miny, maxx, maxy)

    with rasterio.open(pop_tif) as src:
        # Reproject bbox to raster CRS for masking
        from rasterio.warp import transform_geom
        geom_src = transform_geom('EPSG:4326', src.crs, mapping(geom_wgs84))

        out_img, out_transform = mask(src, [geom_src], crop=True, nodata=src.nodata)
        arr = out_img[0].astype(float)

        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        pop_total = float(np.nansum(arr))
        valid_pixels = int(np.sum(~np.isnan(arr)))

        # Save cropped raster
        crop_path = CROPS_DIR / f'{event_id}_pop_crop.tif'
        profile = src.profile.copy()
        profile.update({
            'height': out_img.shape[1],
            'width': out_img.shape[2],
            'transform': out_transform,
        })

        with rasterio.open(crop_path, 'w', **profile) as dst:
            dst.write(out_img)

    # Geodesic area from original bbox in WGS84
    area_m2, _ = geod.geometry_area_perimeter(geom_wgs84)
    area_km2 = abs(area_m2) / 1e6
    density = pop_total / area_km2 if area_km2 > 0 else np.nan

    rows.append({
        'event': event_id,
        'country': country,
        'population_raster': pop_tif.name,
        'total_population': round(pop_total, 2),
        'area_km2': round(area_km2, 4),
        'population_density_km2': round(float(density), 2),
        'valid_pixels': valid_pixels,
        'crop_path': str(crop_path.relative_to(ROOT)).replace('\\', '/'),
    })

df = pd.DataFrame(rows).sort_values('event').reset_index(drop=True)
df

,event,country,population_raster,total_population,area_km2,population_density_km2,valid_pixels,crop_path
0,ahr_valley,germany,deu_pop_2021_CN_100m_R2025A_v1.tif,38365.01,177.0092,216.74,9159,Population_data/cropped_to_events/ahr_valley_p...
1,antalya,turkey,tur_pop_2021_CN_100m_R2025A_v1.tif,465543.15,5868.3958,79.33,106724,Population_data/cropped_to_events/antalya_pop_...
2,emilia,italy,ita_pop_2023_CN_100m_R2025A_v1.tif,15909.53,132.3286,120.23,11262,Population_data/cropped_to_events/emilia_pop_c...
3,evros,greece,grc_pop_2023_CN_100m_R2025A_v1.tif,167135.48,4572.4145,36.55,117438,Population_data/cropped_to_events/evros_pop_cr...
4,serra_de_estrela,portugal,prt_pop_2022_CN_100m_R2025A_v1.tif,69997.87,1320.4370,53.01,68839,Population_data/cropped_to_events/serra_de_est...
5,sindh,pakistan,pak_pop_2022_CN_100m_R2025A_v1.tif,3828128.98,6122.3091,625.28,260953,Population_data/cropped_to_events/sindh_pop_cr...


In [18]:
# Exposure classification for reasoning
# You can adjust thresholds later based on project needs.
def exposure_class(d):
    if np.isnan(d):
        return 'unknown'
    if d < 100:
        return 'low'
    if d < 500:
        return 'medium'
    return 'high'

df['population_exposure_class'] = df['population_density_km2'].apply(exposure_class)
df

,event,country,population_raster,total_population,area_km2,population_density_km2,valid_pixels,crop_path,population_exposure_class
0,ahr_valley,germany,deu_pop_2021_CN_100m_R2025A_v1.tif,38365.01,177.0092,216.74,9159,Population_data/cropped_to_events/ahr_valley_p...,medium
1,antalya,turkey,tur_pop_2021_CN_100m_R2025A_v1.tif,465543.15,5868.3958,79.33,106724,Population_data/cropped_to_events/antalya_pop_...,low
2,emilia,italy,ita_pop_2023_CN_100m_R2025A_v1.tif,15909.53,132.3286,120.23,11262,Population_data/cropped_to_events/emilia_pop_c...,medium
3,evros,greece,grc_pop_2023_CN_100m_R2025A_v1.tif,167135.48,4572.4145,36.55,117438,Population_data/cropped_to_events/evros_pop_cr...,low
4,serra_de_estrela,portugal,prt_pop_2022_CN_100m_R2025A_v1.tif,69997.87,1320.4370,53.01,68839,Population_data/cropped_to_events/serra_de_est...,low
5,sindh,pakistan,pak_pop_2022_CN_100m_R2025A_v1.tif,3828128.98,6122.3091,625.28,260953,Population_data/cropped_to_events/sindh_pop_cr...,high


In [19]:
# Export CSV summary
df.to_csv(OUT_CSV, index=False)
OUT_CSV

WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/population_exposure_summary.csv')

In [20]:
# Export ASL facts for MAS reasoning
lines = []
lines.append('// ----------------------------------------------')
lines.append('// Population exposure indicators')
lines.append('// Auto-generated from population_exposure_from_bbox.ipynb')
lines.append('// ----------------------------------------------\n')

for _, r in df.iterrows():
    e = r['event']
    lines.append(f"total_population({e}, {r['total_population']:.2f}).")
    lines.append(f"population_density_km2({e}, {r['population_density_km2']:.2f}).")
    lines.append(f"population_exposure_class({e}, {r['population_exposure_class']}).")
    lines.append(f"population_valid_pixels({e}, {int(r['valid_pixels'])}).")
    lines.append('')

OUT_ASL.write_text('\n'.join(lines), encoding='utf-8')
OUT_ASL

WindowsPath('c:/Users/Stefano/Desktop/SDAINLP_FinalProject/EO2Explain/beliefs/population_exposure.asl')